# Fashion-MNIST Architectures, Part 2: Linear, MLP, CNN, LSTM/GRU and Transformer

**Author:** Dương Nguyễn Tấn Đạt

## Problem formulation
- **Task:** single-label image classification of fashion products into 10 classes.
- **Input:** a grayscale image $x \in [0, 255]^{28 \times 28}$ (one product per image, centered on a black background).
- **Output:** a class label $\hat{y} \in \{0, \dots, 9\}$, obtained as $\hat{y} = \arg\max_k z_k$ from the logits $z = f_\theta(x) \in \mathbb{R}^{10}$.
- **Prediction unit:** one image.
- **Training objective:** minimize the mean cross-entropy $\mathcal{L}(\theta) = -\frac{1}{N}\sum_{i=1}^{N} \log \mathrm{softmax}(f_\theta(x_i))_{y_i}$.
- **Main difficulty:** several upper-body garment classes (T-shirt/top, Pullover, Coat, Shirt) share almost the same silhouette at $28 \times 28$ resolution.
- **Research question:** how do the data representation (flattened pixels, 2-D feature maps, row sequences, patch tokens) and the resulting inductive bias of each architecture family affect accuracy, cost and failure modes?

## Datasets used
| Role | Dataset |
|---|---|
| Debugging | Not used (MNIST is allowed only for debugging) |
| Main comparison | **Fashion-MNIST** (Zalando Research, MIT license, loaded via `torchvision.datasets.FashionMNIST`) |
| Extension | None |

## Core and extension experiments
| Item | Status |
|---|---|
| Linear, MLP, CNN, LSTM, Transformer | **Core comparison**, all implemented and trained from scratch |
| MLP capacity ablation (MLP-A/B/C) | Recommended ablation |
| GRU vs. LSTM | Extension |
| Transformer without positional encoding | Ablation of the representation |
| CNN at a parameter budget matched to the largest MLP | Extension (matched budget) |
| CNN feature maps, RNN prefix accuracy, attention maps | Extension (interpretability) |

## Pipeline
```
Raw uint8 images -> stratified split -> normalization (+ horizontal flip on train) -> Dataset / DataLoader
-> model (Linear | MLP | CNN | LSTM/GRU | Transformer) -> cross-entropy loss -> AdamW + cosine LR
-> argmax prediction -> checkpoint selection on validation macro-F1 -> one-time test evaluation
-> metrics, curves, error analysis, interpretability
```

> **How to run on Google Colab:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`. With 9 models and 3 training seeds the full run takes roughly 1 to 1.5 hours on a T4. Set `TRAIN_SEEDS = [42]` in Section 1 for a quick debugging run. At the end, the notebook downloads `a1_results.zip` (all numbers, figures and checkpoints).

## Section 1: Reproducibility and Environment
- `SPLIT_SEED` fixes the train/validation partition. It is shared by every model, so all models see exactly the same data.
- `TRAIN_SEEDS` controls weight initialization, shuffling order and augmentation. Each model is trained once per seed so results are reported as mean ± std instead of a single lucky run.
- `set_seed` is called **right before each model is built**, so a model's initial weights do not depend on how many other models were created before it.

In [ ]:
import os
import sys
import time
import json
import random
import hashlib
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from IPython.display import display

SPLIT_SEED = 42                  # one fixed partition for every model
TRAIN_SEEDS = [42, 123, 2026]    # independent training runs per model


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SPLIT_SEED)
# Deterministic cuDNN kernels trade a little speed for run-to-run repeatability
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ENV_INFO = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "numpy": np.__version__,
    "scikit-learn": sklearn.__version__,
    "device": str(device),
    "cuda": torch.version.cuda if torch.cuda.is_available() else None,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if torch.cuda.is_available() else None,
    "split_seed": SPLIT_SEED,
    "train_seeds": TRAIN_SEEDS,
}
for k, v in ENV_INFO.items():
    print(f"{k:>14}: {v}")

FIG_DIR = "./results/figures"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs("./checkpoints", exist_ok=True)
with open("./results/environment.json", "w") as f:
    json.dump(ENV_INFO, f, indent=2)


def save_fig(name, fig=None):
    # Vector copies for the LaTeX report; the notebook still shows the figure inline
    (fig or plt.gcf()).savefig(os.path.join(FIG_DIR, f"{name}.pdf"), bbox_inches="tight")

## Section 2: Data Loading, Split and Leakage Checks
**Source:** Fashion-MNIST, 60,000 official training images and 10,000 official test images, $28 \times 28$ grayscale, 10 classes.

**Split policy:**
- The official **test** set is kept untouched and is evaluated only once per trained model, after checkpoint selection.
- The official training set is split into **train** (5,000 per class = 50,000) and **validation** (1,000 per class = 10,000).
- The split is **stratified by class** and uses `SPLIT_SEED`, so class proportions are identical across splits and the partition is reproducible. Split indices are saved to `results/split_indices.pt`.

**Leakage checks:**
- Train and validation indices are disjoint (asserted).
- Normalization statistics are computed from the **train split only**, so validation and test pixels never influence preprocessing.
- Exact pixel-level duplicates are detected by hashing every image. Duplicates across splits would inflate validation/test scores.

In [ ]:
CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]
NUM_CLASSES = len(CLASS_NAMES)
DATA_DIR = "./data"

raw_train = datasets.FashionMNIST(root=DATA_DIR, train=True, download=True)
raw_test = datasets.FashionMNIST(root=DATA_DIR, train=False, download=True)

# Raw uint8 tensors of shape [N, 28, 28]; working on tensors avoids a PIL round-trip per sample
trainval_images, trainval_labels = raw_train.data, raw_train.targets
test_images, test_labels = raw_test.data, raw_test.targets

VAL_PER_CLASS = 1000
split_gen = torch.Generator().manual_seed(SPLIT_SEED)
train_parts, val_parts = [], []
for c in range(NUM_CLASSES):
    cls_idx = torch.nonzero(trainval_labels == c).squeeze(1)
    cls_idx = cls_idx[torch.randperm(len(cls_idx), generator=split_gen)]
    val_parts.append(cls_idx[:VAL_PER_CLASS])
    train_parts.append(cls_idx[VAL_PER_CLASS:])
train_idx = torch.cat(train_parts).sort().values
val_idx = torch.cat(val_parts).sort().values
test_idx = torch.arange(len(test_labels))

assert len(set(train_idx.tolist()) & set(val_idx.tolist())) == 0, "train/val overlap"
assert len(train_idx) + len(val_idx) == len(trainval_labels)
torch.save({"split_seed": SPLIT_SEED, "train_idx": train_idx, "val_idx": val_idx}, "./results/split_indices.pt")

print(pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Source": ["official train (stratified)", "official train (stratified)", "official test"],
    "Samples": [len(train_idx), len(val_idx), len(test_idx)],
}).to_string(index=False))

# Normalization statistics from the train split only
train_pixels = trainval_images[train_idx].float().div(255.0)
NORM_MEAN = train_pixels.mean().item()
NORM_STD = train_pixels.std().item()
del train_pixels
print(f"\nNormalization stats (train split only): mean = {NORM_MEAN:.4f}, std = {NORM_STD:.4f}")

# Exact-duplicate detection via per-image hashes
def image_hashes(images):
    return np.array([hashlib.md5(img.numpy().tobytes()).hexdigest() for img in images])

trainval_hashes = image_hashes(trainval_images)
test_hashes = image_hashes(test_images)
split_hashes = {
    "Train": trainval_hashes[train_idx.numpy()],
    "Validation": trainval_hashes[val_idx.numpy()],
    "Test": test_hashes,
}
dup_stats = {}
print("\nExact duplicate images:")
for name, h in split_hashes.items():
    dup_stats[f"within {name}"] = len(h) - len(set(h))
    print(f"  within {name:<10}: {dup_stats[f'within {name}']}")
for a, b in [("Train", "Validation"), ("Train", "Test"), ("Validation", "Test")]:
    dup_stats[f"{b} in {a}"] = int(np.isin(split_hashes[b], split_hashes[a]).sum())
    print(f"  {b} images whose exact copy is in {a}: {dup_stats[f'{b} in {a}']}")

## Section 3: Exploratory Data Analysis
The analysis covers class distribution, input size, imbalance and representative samples. We also inspect pixel statistics, because several of the models see raw flattened pixels and nothing else.

In [ ]:
# 3.1 Input size and value range
sample = trainval_images[0]
print(f"Image tensor shape : {tuple(sample.shape)} (H x W), 1 channel, dtype {sample.dtype}")
print(f"Flattened dim      : {sample.numel()} features per image")
print(f"Pixel value range  : [{trainval_images.min().item()}, {trainval_images.max().item()}]")
background_pct = (trainval_images == 0).float().mean().item() * 100
print(f"Background pixels  : {background_pct:.1f}% of all train/val pixels are exactly 0")
storage_mb = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(DATA_DIR) for f in fs) / 1024**2
print(f"On-disk size       : {storage_mb:.1f} MB (raw IDX files + archives in {DATA_DIR})")

# 3.2 Class distribution per split and imbalance ratio
dist = pd.DataFrame({
    "Train": np.bincount(trainval_labels[train_idx].numpy(), minlength=NUM_CLASSES),
    "Validation": np.bincount(trainval_labels[val_idx].numpy(), minlength=NUM_CLASSES),
    "Test": np.bincount(test_labels.numpy(), minlength=NUM_CLASSES),
}, index=CLASS_NAMES)
print("\nSamples per class:")
display(dist)
imbalance = (dist.max() / dist.min()).rename("Imbalance ratio (max/min)")
print(imbalance.to_string())

ax = dist.plot(kind="bar", figsize=(12, 4), width=0.8, color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_title("Class distribution per split")
ax.set_ylabel("Number of images")
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
ax.grid(axis="y", linestyle=":", alpha=0.6)
plt.tight_layout()
save_fig("eda_class_distribution")
plt.show()

In [ ]:
# 3.3 Representative samples (8 random training images per class) and the per-class mean image
samples_per_class = 8
eda_gen = torch.Generator().manual_seed(SPLIT_SEED)
train_labels_split = trainval_labels[train_idx]

fig, axes = plt.subplots(NUM_CLASSES, samples_per_class + 1, figsize=(11, 13))
for c in range(NUM_CLASSES):
    cls_pool = train_idx[train_labels_split == c]
    picks = cls_pool[torch.randperm(len(cls_pool), generator=eda_gen)[:samples_per_class]]
    for j, idx in enumerate(picks):
        axes[c, j].imshow(trainval_images[idx], cmap="gray")
        axes[c, j].axis("off")
    axes[c, 0].set_title(CLASS_NAMES[c], fontsize=9, loc="left")
    mean_img = trainval_images[cls_pool].float().mean(0)
    axes[c, -1].imshow(mean_img, cmap="magma")
    axes[c, -1].axis("off")
axes[0, -1].set_title("class mean", fontsize=9)
plt.suptitle("Representative training samples (left) and per-class mean image (right)", fontsize=13)
plt.tight_layout()
save_fig("eda_samples")
plt.show()

In [ ]:
# 3.4 Pixel statistics: global intensity histogram and per-class image statistics
train_imgs_float = trainval_images[train_idx].float() / 255.0
per_image = pd.DataFrame({
    "class": [CLASS_NAMES[i] for i in train_labels_split.tolist()],
    "mean intensity": train_imgs_float.mean(dim=(1, 2)).numpy(),
    "foreground ratio": (train_imgs_float > 0).float().mean(dim=(1, 2)).numpy(),
})

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
axes[0].hist(train_imgs_float.flatten().numpy(), bins=50, color="#4C72B0", log=True)
axes[0].set_title("Pixel intensity histogram (train, log scale)")
axes[0].set_xlabel("Intensity in [0, 1]")
axes[0].set_ylabel("Pixel count")
sns.boxplot(data=per_image, x="class", y="mean intensity", ax=axes[1], order=CLASS_NAMES, fliersize=1)
axes[1].set_title("Mean intensity per image")
sns.boxplot(data=per_image, x="class", y="foreground ratio", ax=axes[2], order=CLASS_NAMES, fliersize=1)
axes[2].set_title("Fraction of non-zero pixels per image")
for ax in axes[1:]:
    ax.tick_params(axis="x", rotation=40)
    ax.set_xlabel("")
plt.tight_layout()
save_fig("eda_pixel_stats")
plt.show()
del train_imgs_float

eda_class_stats = per_image.groupby("class")[["mean intensity", "foreground ratio"]].median().loc[CLASS_NAMES].round(3)
print(eda_class_stats.to_string())

### EDA observations
- **Balance:** every class has exactly 5,000 / 1,000 / 1,000 images in train / validation / test (imbalance ratio 1.0). Accuracy and macro-F1 are therefore expected to be close, and no class re-weighting or resampling is needed.
- **Input size:** every image is $1 \times 28 \times 28$ (784 features when flattened), uint8 in $[0, 255]$. A large share of pixels is exactly 0 (black background), so many input dimensions carry little information.
- **Visual similarity:** the class mean images and the per-image statistics show that T-shirt/top, Pullover, Coat and Shirt have nearly the same silhouette and similar brightness / coverage. These classes differ mainly in fine details (collar, buttons, sleeve length, texture), which is the main expected source of errors.
- **Easier classes:** Trouser, Bag and the footwear classes have distinctive shapes or coverage, so they should be separable even by a linear model.
- **Bias and limitations:** images are studio product photos, centered, background-removed and downscaled to $28 \times 28$. Models trained here are not expected to transfer to real-world photos without additional data.
- **Duplicates:** see the duplicate counts printed in Section 2. Any cross-split duplicates are reported, not removed, because the official test set is kept unchanged for comparability with the literature.

## Section 4: Preprocessing, Dataset and DataLoader
- **Normalization (all splits):** $x' = (x / 255 - \mu) / \sigma$ with $\mu, \sigma$ from the train split. Zero-centered inputs keep the first linear layer's pre-activations well scaled, which speeds up optimization.
- **Augmentation (train only):** `RandomHorizontalFlip(p=0.5)`. Fashion items are roughly left-right symmetric, so a mirrored image is still a valid sample of the same class. Vertical flips, large rotations and aggressive crops are avoided because they create unrealistic images, and flattened-pixel models have no translation invariance to absorb large shifts.
- **Dataset design:** `FashionMNISTTensorDataset` serves samples from the in-memory uint8 tensor and applies the transforms with tensor ops. This avoids the per-sample PIL conversion of `torchvision.transforms.ToTensor`, which was the bottleneck in an earlier version of this pipeline (it made all models look equally slow).
- **Batching:** batch size 64 for training (shuffled, with a seeded `torch.Generator`), 512 for evaluation (order fixed). `num_workers=0` keeps augmentation randomness tied to the global seed, and tensor-based loading is fast enough not to need workers.

In [ ]:
class FashionMNISTTensorDataset(Dataset):
    """Returns normalized [1, 28, 28] float tensors and integer labels for one split."""

    def __init__(self, images, labels, indices, mean, std, hflip_p=0.0):
        # Keep only this split's samples so a dataset can never index into another split
        self.images = images[indices]
        self.labels = labels[indices]
        self.mean = mean
        self.std = std
        self.hflip_p = hflip_p

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        x = self.images[i].unsqueeze(0).float().div_(255.0)
        if self.hflip_p > 0 and torch.rand(1).item() < self.hflip_p:
            x = x.flip(-1)
        x = (x - self.mean) / self.std
        return x, int(self.labels[i])


BATCH_SIZE = 64
EVAL_BATCH_SIZE = 512
HFLIP_P = 0.5

train_dataset = FashionMNISTTensorDataset(trainval_images, trainval_labels, train_idx, NORM_MEAN, NORM_STD, hflip_p=HFLIP_P)
val_dataset = FashionMNISTTensorDataset(trainval_images, trainval_labels, val_idx, NORM_MEAN, NORM_STD)
test_dataset = FashionMNISTTensorDataset(test_images, test_labels, test_idx, NORM_MEAN, NORM_STD)


def make_train_loader(seed):
    return DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
                      pin_memory=torch.cuda.is_available(),
                      generator=torch.Generator().manual_seed(seed))


val_loader = DataLoader(val_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

print(f"Train: {len(train_dataset)} samples / {len(make_train_loader(0))} batches of {BATCH_SIZE}")
print(f"Val  : {len(val_dataset)} samples / {len(val_loader)} batches of {EVAL_BATCH_SIZE}")
print(f"Test : {len(test_dataset)} samples / {len(test_loader)} batches of {EVAL_BATCH_SIZE}")

In [ ]:
# One example training batch after preprocessing
set_seed(SPLIT_SEED)
xb, yb = next(iter(make_train_loader(SPLIT_SEED)))
print(f"Batch images: shape {tuple(xb.shape)}, dtype {xb.dtype}, mean {xb.mean():.3f}, std {xb.std():.3f}, "
      f"min {xb.min():.3f}, max {xb.max():.3f}")
print(f"Batch labels: shape {tuple(yb.shape)}, dtype {yb.dtype}, first 16 = {yb[:16].tolist()}")

fig, axes = plt.subplots(2, 8, figsize=(14, 4.8))
for i, ax in enumerate(axes.flat):
    ax.imshow((xb[i, 0] * NORM_STD + NORM_MEAN).clamp(0, 1), cmap="gray")
    ax.set_title(CLASS_NAMES[yb[i]], fontsize=9)
    ax.axis("off")
plt.suptitle("First 16 images of a preprocessed training batch (de-normalized for display, flips included)")
plt.tight_layout()
save_fig("example_batch")
plt.show()

## Section 5: Models
Every model is implemented from scratch on top of `torch.nn` layers and trained from random initialization. No pretrained weights are used.

| Model | Family | Input representation | Role |
|---|---|---|---|
| Linear | Linear / softmax | flattened pixels (784) | Core |
| MLP-A / MLP-B / MLP-C | MLP | flattened pixels (784) | Core + capacity ablation (E2) |
| CNN | Convolutional | 2-D image $1 \times 28 \times 28$ | Core (E3), matched budget to MLP-C |
| LSTM | Recurrent | sequence of 28 rows, 28 pixels each | Core (E3) |
| GRU | Recurrent | sequence of 28 rows, 28 pixels each | Extension (E4) |
| Transformer | Attention | 49 patch tokens of $4 \times 4$ pixels + CLS | Core (E3) |
| Transformer, no pos. enc. | Attention | 49 patch tokens, positions removed | Ablation (E5) |

**Linear / softmax classifier** - a single affine map $z = Wx + b$ with $W \in \mathbb{R}^{10 \times 784}$ (7,850 parameters). Each row of $W$ is a "template" image for one class. The model outputs raw logits: `nn.CrossEntropyLoss` applies log-softmax internally, so **no softmax is applied before the loss** (that would squash gradients and is numerically unstable).

**MLP** - hidden blocks of `Linear -> BatchNorm1d -> ReLU -> Dropout(0.2)` and a linear output layer. ReLU adds the non-linearity and does not saturate for positive inputs. BatchNorm keeps hidden activations well scaled. Dropout zeroes 20% of hidden units during training to reduce co-adaptation. Weight decay ($10^{-4}$) shrinks all weights. Three sizes (128; 256; 512-256 hidden units) form the capacity ablation.

**CNN** - two convolutional stages, each `Conv3x3 -> BN -> ReLU -> Conv3x3 -> BN -> ReLU -> MaxPool2x2`, followed by an MLP head.
- A **convolution** slides a small learned $3 \times 3$ kernel over the image and computes a weighted sum at every position. The same kernel is reused everywhere (**weight sharing**), so a pattern such as an edge or a collar corner is detected wherever it appears, with far fewer parameters than a fully connected layer.
- Each kernel produces one **feature map**: a 2-D grid that shows where its pattern occurs. Stage 1 outputs 32 maps of $28 \times 28$ (pooled to $14 \times 14$), stage 2 outputs 64 maps of $14 \times 14$ (pooled to $7 \times 7$). Deeper maps combine simpler ones into larger, more abstract parts.
- **Max-pooling** keeps the largest activation in each $2 \times 2$ window. It halves the resolution, enlarges the receptive field of later layers and makes the features tolerant to small shifts.
- The head flattens the $64 \times 7 \times 7$ maps, applies dropout and two linear layers. The total parameter count is close to MLP-C, so E3 compares the two at a matched budget.

**LSTM / GRU** - the image is read as a sequence from top to bottom.
- **Timestep definition:** timestep $t \in \{1, \dots, 28\}$ is pixel row $t$. **Input size:** 28 (the pixels of that row).
- **Hidden representation:** a 2-layer LSTM (or GRU) with 128 hidden units per layer. After reading row $t$, the hidden state $h_t \in \mathbb{R}^{128}$ summarizes rows $1..t$. The LSTM keeps an additional cell state $c_t$ controlled by input, forget and output gates. The GRU merges these into update and reset gates and has about 25% fewer parameters.
- **Classification:** the top-layer hidden state after the last row, $h_{28}$, is passed to a linear layer. Dropout 0.2 is applied between the two recurrent layers.

**Transformer (ViT-style)** - the image is cut into $7 \times 7 = 49$ non-overlapping patches of $4 \times 4$ pixels.
- **Token embedding:** each flattened patch ($16$ values) is projected to $d = 64$ dimensions by a shared linear layer. A learnable `[CLS]` token is prepended, giving 50 tokens.
- **Positional encoding:** a learned vector per position (50 x 64) is added to the tokens. Without it, self-attention is permutation-invariant, so the model could not tell where a patch came from (tested in E5).
- **Attention:** 4 pre-norm encoder blocks, each with 4-head self-attention and a feed-forward layer of width 128 (GELU). Inside a head, each token is projected to a query $q$, key $k$ and value $v$ of size 16. The output for a token is $\mathrm{softmax}(q K^\top / \sqrt{16}) V$, i.e. a weighted average of all value vectors, with weights given by query-key similarity. **Input:** a sequence of 50 tokens of size 64. **Output:** a sequence of the same shape in which every token has mixed in information from all others. The final `[CLS]` token is normalized and classified by a linear layer.
- Dropout 0.1 is used inside the Transformer, the usual default for attention models.

In [ ]:
class LinearClassifier(nn.Module):
    def __init__(self, input_dim=28 * 28, num_classes=NUM_CLASSES):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x.flatten(1))  # raw logits, no softmax


class MLP(nn.Module):
    def __init__(self, layer_dims, dropout_rate=0.2):
        super().__init__()
        layers = []
        for in_dim, out_dim in zip(layer_dims[:-2], layer_dims[1:-1]):
            layers += [nn.Linear(in_dim, out_dim), nn.BatchNorm1d(out_dim), nn.ReLU(), nn.Dropout(dropout_rate)]
        layers.append(nn.Linear(layer_dims[-2], layer_dims[-1]))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x.flatten(1))


def conv_stage(c_in, c_out):
    # Two 3x3 convolutions keep the resolution (padding=1); max-pooling then halves it.
    # Convolutions have no bias because the following BatchNorm adds its own shift.
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(),
        nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(),
        nn.MaxPool2d(2),
    )


class CNN(nn.Module):
    def __init__(self, channels=(32, 64), hidden=128, dropout=0.2):
        super().__init__()
        self.stage1 = conv_stage(1, channels[0])            # [B, 32, 14, 14]
        self.stage2 = conv_stage(channels[0], channels[1])  # [B, 64, 7, 7]
        self.head = nn.Sequential(
            nn.Flatten(), nn.Dropout(dropout),
            nn.Linear(channels[1] * 7 * 7, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, NUM_CLASSES),
        )

    def forward(self, x):
        return self.head(self.stage2(self.stage1(x)))


class RowRNN(nn.Module):
    """Reads the image top to bottom: timestep t is pixel row t (28 values)."""

    def __init__(self, cell="lstm", hidden=128, num_layers=2, dropout=0.2):
        super().__init__()
        rnn_cls = {"lstm": nn.LSTM, "gru": nn.GRU}[cell]
        self.rnn = rnn_cls(input_size=28, hidden_size=hidden, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.head = nn.Linear(hidden, NUM_CLASSES)

    def forward(self, x, all_steps=False):
        seq = x.squeeze(1)       # [B, 28 timesteps, 28 features]
        out, _ = self.rnn(seq)   # [B, 28, hidden]: top-layer hidden state after each row
        if all_steps:
            return self.head(out)  # logits after reading rows 1..t (analysis only)
        return self.head(out[:, -1])


class EncoderBlock(nn.Module):
    """Pre-norm Transformer encoder block that can keep its attention weights for inspection."""

    def __init__(self, dim, heads, ffn_dim, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, ffn_dim), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(ffn_dim, dim), nn.Dropout(dropout))
        self.last_attn = None

    def forward(self, x, keep_attn=False):
        h = self.norm1(x)
        a, w = self.attn(h, h, h, need_weights=keep_attn, average_attn_weights=True)
        if keep_attn:
            self.last_attn = w.detach()  # [B, tokens, tokens], averaged over heads
        x = x + self.drop(a)
        return x + self.ffn(self.norm2(x))


class PatchTransformer(nn.Module):
    def __init__(self, patch=4, dim=64, depth=4, heads=4, ffn_dim=128, dropout=0.1, use_pos=True):
        super().__init__()
        self.patch = patch
        n_patches = (28 // patch) ** 2
        self.proj = nn.Linear(patch * patch, dim)  # token embedding of a flattened patch
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.zeros(1, n_patches + 1, dim)) if use_pos else None
        nn.init.trunc_normal_(self.cls, std=0.02)
        if self.pos is not None:
            nn.init.trunc_normal_(self.pos, std=0.02)
        self.blocks = nn.ModuleList([EncoderBlock(dim, heads, ffn_dim, dropout) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, NUM_CLASSES)

    def patchify(self, x):
        p = self.patch
        x = x.unfold(2, p, p).unfold(3, p, p)  # [B, 1, 7, 7, p, p]
        return x.reshape(x.size(0), -1, p * p)  # [B, 49, p*p], patches in row-major order

    def forward(self, x, keep_attn=False):
        tokens = self.proj(self.patchify(x))
        tokens = torch.cat([self.cls.expand(x.size(0), -1, -1), tokens], dim=1)
        if self.pos is not None:
            tokens = tokens + self.pos
        for block in self.blocks:
            tokens = block(tokens, keep_attn)
        return self.head(self.norm(tokens[:, 0]))


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# key -> (display name, factory). Factories let every run build a fresh model right after seeding.
MODEL_SPECS = {
    "linear": ("Linear (784-10)", lambda: LinearClassifier()),
    "mlp_a": ("MLP-A (784-128-10)", lambda: MLP([784, 128, 10])),
    "mlp_b": ("MLP-B (784-256-10)", lambda: MLP([784, 256, 10])),
    "mlp_c": ("MLP-C (784-512-256-10)", lambda: MLP([784, 512, 256, 10])),
    "cnn": ("CNN (32-64 channels)", lambda: CNN()),
    "lstm": ("LSTM (rows, 2x128)", lambda: RowRNN("lstm")),
    "gru": ("GRU (rows, 2x128)", lambda: RowRNN("gru")),
    "vit": ("Transformer (4x4 patches)", lambda: PatchTransformer()),
    "vit_nopos": ("Transformer, no pos. enc.", lambda: PatchTransformer(use_pos=False)),
}
DISPLAY = {k: v[0] for k, v in MODEL_SPECS.items()}
SHORT = {"linear": "Linear", "mlp_a": "MLP-A", "mlp_b": "MLP-B", "mlp_c": "MLP-C", "cnn": "CNN",
         "lstm": "LSTM", "gru": "GRU", "vit": "ViT", "vit_nopos": "ViT-noPE"}
FAMILY = {"linear": "Linear", "mlp_a": "MLP", "mlp_b": "MLP", "mlp_c": "MLP", "cnn": "CNN",
          "lstm": "RNN", "gru": "RNN", "vit": "Transformer", "vit_nopos": "Transformer"}
REPRESENTATION = {"linear": "flattened pixels (784)", "mlp_a": "flattened pixels (784)", "mlp_b": "flattened pixels (784)",
                  "mlp_c": "flattened pixels (784)", "cnn": "2-D image (1 × 28 × 28)",
                  "lstm": "28 rows × 28 pixels", "gru": "28 rows × 28 pixels",
                  "vit": "49 patches (4 × 4) + CLS", "vit_nopos": "49 patches, no positions"}
ROLE = {"linear": "Core", "mlp_a": "Core / E2", "mlp_b": "Core / E2", "mlp_c": "Core / E2", "cnn": "Core",
        "lstm": "Core", "gru": "Extension (E4)", "vit": "Core", "vit_nopos": "Ablation (E5)"}
COLORS = dict(zip(MODEL_SPECS, sns.color_palette("tab10", len(MODEL_SPECS))))

param_counts = {key: count_parameters(factory()) for key, (_, factory) in MODEL_SPECS.items()}
# Sanity checks against closed-form counts (weights + biases, BatchNorm gamma + beta)
assert param_counts["linear"] == 784 * 10 + 10
assert param_counts["mlp_a"] == (784 * 128 + 128) + 2 * 128 + (128 * 10 + 10)
assert param_counts["cnn"] == (9 * (1 * 32 + 32 * 32 + 32 * 64 + 64 * 64)) + 2 * (32 + 32 + 64 + 64) + (3136 * 128 + 128) + (128 * 10 + 10)
assert param_counts["lstm"] == 4 * (28 * 128 + 128 * 128 + 2 * 128) + 4 * (2 * 128 * 128 + 2 * 128) + (128 * 10 + 10)

model_table = pd.DataFrame({
    "Model": [DISPLAY[k] for k in MODEL_SPECS], "Family": [FAMILY[k] for k in MODEL_SPECS],
    "Representation": [REPRESENTATION[k] for k in MODEL_SPECS], "Role": [ROLE[k] for k in MODEL_SPECS],
    "Parameters": [f"{param_counts[k]:,}" for k in MODEL_SPECS],
})
display(model_table)

# Shape walk-through for one batch, to document what each architecture sees
with torch.no_grad():
    probe = torch.zeros(2, 1, 28, 28)
    cnn_probe = CNN().eval()
    print("CNN    :", tuple(probe.shape), "-> stage1", tuple(cnn_probe.stage1(probe).shape),
          "-> stage2", tuple(cnn_probe.stage2(cnn_probe.stage1(probe)).shape), "-> logits", tuple(cnn_probe(probe).shape))
    print("LSTM   :", tuple(probe.shape), "-> sequence", tuple(probe.squeeze(1).shape), "-> logits", tuple(RowRNN().eval()(probe).shape))
    vit_probe = PatchTransformer().eval()
    print("ViT    :", tuple(probe.shape), "-> patches", tuple(vit_probe.patchify(probe).shape),
          "-> tokens", (2, 50, 64), "-> logits", tuple(vit_probe(probe).shape))

## Section 6: Training, Evaluation and Timing Engine
- **Loss accumulation:** batch losses are weighted by batch size, so the epoch loss is the exact per-sample mean even when the last batch is smaller. Losses are summed on the device and synchronized once per epoch.
- **Train metrics** are collected in `model.train()` mode, i.e. with dropout active and on flipped images. They are therefore a slightly pessimistic estimate and not directly comparable with validation metrics.
- **Training time** = wall-clock time of the training passes only (data loading + forward + backward + update), excluding validation. The GPU is synchronized before reading the clock, and every architecture is warmed up once before the experiments so CUDA / cuDNN initialization is not charged to the first run.
- **Inference time** is measured on a batch already resident on the device, so it reflects model compute only. We report latency at batch size 1 (µs per image, the online setting) and per-image time at batch size 512 (the throughput setting).

In [ ]:
def sync():
    if device.type == "cuda":
        torch.cuda.synchronize()


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    loss_sum = torch.zeros((), device=device)
    preds, targets = [], []
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.detach() * x.size(0)
        preds.append(logits.argmax(1))
        targets.append(y)
    preds = torch.cat(preds).cpu().numpy()
    targets = torch.cat(targets).cpu().numpy()
    return {
        "loss": loss_sum.item() / len(targets),
        "acc": float((preds == targets).mean()),
        "f1": f1_score(targets, preds, average="macro"),
    }


@torch.no_grad()
def evaluate(model, loader, criterion, return_outputs=False):
    model.eval()
    loss_sum = torch.zeros((), device=device)
    logits_all, targets = [], []
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits = model(x)
        loss_sum += criterion(logits, y) * x.size(0)
        logits_all.append(logits)
        targets.append(y)
    logits_all = torch.cat(logits_all)
    preds = logits_all.argmax(1).cpu().numpy()
    targets = torch.cat(targets).cpu().numpy()
    out = {
        "loss": loss_sum.item() / len(targets),
        "acc": float((preds == targets).mean()),
        "f1": f1_score(targets, preds, average="macro"),
    }
    if return_outputs:
        out.update(preds=preds, targets=targets, probs=F.softmax(logits_all, dim=1).cpu().numpy())
    return out


@torch.inference_mode()
def measure_inference_ms(model, batch_size, n_iters, warmup=20):
    """Average milliseconds per image for a forward pass on a device-resident batch."""
    model.eval()
    x = torch.randn(batch_size, 1, 28, 28, device=device)
    for _ in range(warmup):
        model(x)
    sync()
    start = time.perf_counter()
    for _ in range(n_iters):
        model(x)
    sync()
    return (time.perf_counter() - start) / (n_iters * batch_size) * 1000.0


def warm_up_device():
    # One-time CUDA / cuBLAS / cuDNN initialization for every architecture, so no run pays for it
    x = torch.randn(BATCH_SIZE, 1, 28, 28, device=device)
    y = torch.randint(0, NUM_CLASSES, (BATCH_SIZE,), device=device)
    for _, factory in MODEL_SPECS.values():
        set_seed(0)
        m = factory().to(device)
        opt = torch.optim.AdamW(m.parameters())
        for _ in range(5):
            opt.zero_grad()
            F.cross_entropy(m(x), y).backward()
            opt.step()
    sync()

## Section 7: Experimental Setup
| Item | Setting |
|---|---|
| Optimizer | AdamW, lr $10^{-3}$, weight decay $10^{-4}$ |
| Scheduler | CosineAnnealingLR from $10^{-3}$ to $10^{-5}$ over 20 epochs (stepped per epoch) |
| Epochs / batch size | 20 / 64 |
| Loss | `nn.CrossEntropyLoss` on raw logits |
| Regularization | weight decay (all); BatchNorm + Dropout 0.2 (MLP, CNN); dropout 0.2 between LSTM/GRU layers; dropout 0.1 (Transformer); horizontal flip (all) |
| Checkpoint rule | epoch with the highest **validation macro-F1** (ties keep the earlier epoch) |
| Test protocol | test set evaluated once per run, after checkpoint selection; never used to choose anything |
| Seeds | split seed 42 (shared); training seeds 42, 123, 2026 |
| Hyperparameter selection | one shared setting for all models, not tuned per model, so that differences come from the architecture |

Every experiment uses the same split, preprocessing, optimizer, schedule, epochs, seeds and metrics. **Decision criterion** for every comparison: a difference in mean test macro-F1 is called *clear* if it is larger than the sum of the two models' standard deviations over seeds; otherwise it is *within seed noise*.

| ID | Hypothesis | Changed factor | Metrics |
|---|---|---|---|
| E1 | A non-linear MLP beats the linear classifier on flattened pixels, because several classes are not linearly separable in pixel space. | Linear vs. MLP | acc, macro-F1, params, time |
| E2 | Larger / deeper MLPs help with diminishing returns and a growing train-validation gap. | MLP width / depth | macro-F1, gap |
| E3 | Representations that keep spatial structure (2-D maps, row sequences, positioned patches) beat flattened pixels; the CNN, whose locality and weight-sharing bias matches the task best, is the strongest. | input representation / family | macro-F1, params, time, per-class F1 |
| E4 | GRU matches LSTM with fewer parameters on 28-step sequences. | recurrent cell | macro-F1, params, time |
| E5 | Removing positional encoding hurts the Transformer, because attention alone cannot tell where a patch is. | positional encoding on / off | macro-F1 |

In [ ]:
EPOCHS = 20
INITIAL_LR = 1e-3
MIN_LR = 1e-5
WEIGHT_DECAY = 1e-4
criterion = nn.CrossEntropyLoss()

CONFIG = {
    "epochs": EPOCHS, "lr": INITIAL_LR, "min_lr": MIN_LR, "weight_decay": WEIGHT_DECAY,
    "batch_size": BATCH_SIZE, "eval_batch_size": EVAL_BATCH_SIZE, "hflip_p": HFLIP_P,
    "optimizer": "AdamW", "scheduler": "CosineAnnealingLR", "selection_metric": "val_macro_f1",
    "norm_mean": NORM_MEAN, "norm_std": NORM_STD, "models": DISPLAY, "parameters": param_counts, **ENV_INFO,
}
with open("./results/config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)


def checkpoint_path(key, seed):
    return f"./checkpoints/{key}_seed{seed}_best.pt"


def fit(key, seed):
    set_seed(seed)
    model = MODEL_SPECS[key][1]().to(device)
    train_loader = make_train_loader(seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=INITIAL_LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LR)

    hist = {k: [] for k in ["lr", "train_loss", "train_acc", "train_f1", "val_loss", "val_acc", "val_f1", "epoch_time"]}
    best = {"val_f1": -1.0}
    for epoch in range(1, EPOCHS + 1):
        hist["lr"].append(optimizer.param_groups[0]["lr"])  # LR actually used in this epoch
        sync()
        t0 = time.perf_counter()
        tr = train_epoch(model, train_loader, criterion, optimizer)
        sync()
        hist["epoch_time"].append(time.perf_counter() - t0)
        va = evaluate(model, val_loader, criterion)
        scheduler.step()

        for split, m in [("train", tr), ("val", va)]:
            for metric in ["loss", "acc", "f1"]:
                hist[f"{split}_{metric}"].append(m[metric])

        if va["f1"] > best["val_f1"]:
            best = {"epoch": epoch, "val_f1": va["f1"], "val_acc": va["acc"], "val_loss": va["loss"], "train_acc": tr["acc"]}
            torch.save(model.state_dict(), checkpoint_path(key, seed))

        if epoch % 5 == 0 or epoch == 1:
            print(f"  [{SHORT[key]} | seed {seed}] epoch {epoch:02d}/{EPOCHS} lr {hist['lr'][-1]:.6f} | "
                  f"train loss {tr['loss']:.4f} acc {tr['acc']*100:.2f}% | "
                  f"val loss {va['loss']:.4f} acc {va['acc']*100:.2f}% F1 {va['f1']:.4f} | {hist['epoch_time'][-1]:.1f}s/epoch")

    model.load_state_dict(torch.load(checkpoint_path(key, seed), map_location=device, weights_only=True))
    return model, hist, best


warm_up_device()
runs, histories = [], {}
experiment_start = time.perf_counter()
for key in MODEL_SPECS:
    print("=" * 110)
    print(f"{DISPLAY[key]}  ({param_counts[key]:,} parameters, {REPRESENTATION[key]})")
    print("=" * 110)
    for seed in TRAIN_SEEDS:
        model, hist, best = fit(key, seed)
        test = evaluate(model, test_loader, criterion)
        runs.append({
            "key": key, "Model": DISPLAY[key], "seed": seed, "Parameters": param_counts[key],
            "Best epoch": best["epoch"],
            "Train acc @best (%)": best["train_acc"] * 100,
            "Val acc @best (%)": best["val_acc"] * 100,
            "Val macro-F1": best["val_f1"],
            "Test acc (%)": test["acc"] * 100,
            "Test macro-F1": test["f1"],
            "Train time (s)": sum(hist["epoch_time"]),
            "Latency bs=1 (ms/img)": measure_inference_ms(model, batch_size=1, n_iters=300),
            "Time bs=512 (ms/img)": measure_inference_ms(model, batch_size=512, n_iters=30),
            "Checkpoint": checkpoint_path(key, seed),
        })
        histories[(key, seed)] = hist
        r = runs[-1]
        print(f"  -> seed {seed}: best epoch {r['Best epoch']}, val F1 {r['Val macro-F1']:.4f}, "
              f"test acc {r['Test acc (%)']:.2f}%, test F1 {r['Test macro-F1']:.4f}, train time {r['Train time (s)']:.1f}s\n")

df_runs = pd.DataFrame(runs)
df_runs.to_csv("./results/runs.csv", index=False)
with open("./results/histories.json", "w") as f:
    json.dump({f"{k}_seed{s}": h for (k, s), h in histories.items()}, f)
print(f"All runs finished in {(time.perf_counter() - experiment_start) / 60:.1f} min. "
      "Saved results/runs.csv, results/histories.json, results/config.json and checkpoints/*.pt")

## Section 8: Quantitative Results
All numbers are mean ± std over the training seeds. Validation numbers are at the selected checkpoint (same epoch for accuracy and F1). The train-validation gap uses train-mode accuracy, so it can be negative for strongly regularized models.

In [ ]:
METRIC_COLS = ["Val macro-F1", "Test acc (%)", "Test macro-F1", "Train acc @best (%)", "Val acc @best (%)",
               "Best epoch", "Train time (s)", "Latency bs=1 (ms/img)", "Time bs=512 (ms/img)"]
agg = df_runs.groupby("key", sort=False)[METRIC_COLS].agg(["mean", "std"])
agg[("Gap train-val (pp)", "mean")] = (df_runs.assign(gap=df_runs["Train acc @best (%)"] - df_runs["Val acc @best (%)"])
                                        .groupby("key", sort=False)["gap"].mean())
MODELS = list(agg.index)


def fmt(col, digits, scale=1.0):
    mean, std = agg[(col, "mean")] * scale, agg[(col, "std")].fillna(0) * scale
    return mean.map(f"{{:.{digits}f}}".format) + " ± " + std.map(f"{{:.{digits}f}}".format)


summary = pd.DataFrame({
    "Model": [DISPLAY[k] for k in MODELS],
    "Role": [ROLE[k] for k in MODELS],
    "Params": [f"{param_counts[k]:,}" for k in MODELS],
    "Test acc (%)": fmt("Test acc (%)", 2).values,
    "Test macro-F1": fmt("Test macro-F1", 4).values,
    "Val macro-F1": fmt("Val macro-F1", 4).values,
    "Best epoch": fmt("Best epoch", 1).values,
    "Train-val gap (pp)": agg[("Gap train-val (pp)", "mean")].round(2).values,
    "Train time (s)": fmt("Train time (s)", 1).values,
    "Latency bs=1 (µs/img)": fmt("Latency bs=1 (ms/img)", 1, scale=1000).values,
    "bs=512 (µs/img)": fmt("Time bs=512 (ms/img)", 2, scale=1000).values,
})
summary.to_csv("./results/summary.csv", index=False)
display(summary)

# Main five-family comparison: the variant of each family with the best mean validation macro-F1
family_best = {}
for key in MODELS:
    fam = FAMILY[key]
    if fam not in family_best or agg.loc[key, ("Val macro-F1", "mean")] > agg.loc[family_best[fam], ("Val macro-F1", "mean")]:
        family_best[fam] = key
family_table = pd.DataFrame({
    "Family": list(family_best), "Selected variant (by val F1)": [DISPLAY[k] for k in family_best.values()],
    "Test macro-F1": [fmt("Test macro-F1", 4)[k] for k in family_best.values()],
    "Params": [f"{param_counts[k]:,}" for k in family_best.values()],
}).sort_values("Test macro-F1", ascending=False)
print("Five-family comparison (variant selected on validation):")
display(family_table)

# Decision criterion: mean gap vs. the sum of standard deviations
PAIRS = [("linear", "mlp_a", "E1"), ("mlp_a", "mlp_b", "E2"), ("mlp_b", "mlp_c", "E2"),
         ("mlp_c", "cnn", "E3"), ("mlp_c", "lstm", "E3"), ("mlp_c", "vit", "E3"),
         ("lstm", "cnn", "E3"), ("vit", "cnn", "E3"), ("lstm", "vit", "E3"),
         ("lstm", "gru", "E4"), ("vit_nopos", "vit", "E5")]
pairwise = []
print("Pairwise test macro-F1 comparisons (B - A; criterion: |diff| > std_A + std_B):")
for a, b, exp in PAIRS:
    diff = agg.loc[b, ("Test macro-F1", "mean")] - agg.loc[a, ("Test macro-F1", "mean")]
    noise = np.nan_to_num(agg.loc[a, ("Test macro-F1", "std")] + agg.loc[b, ("Test macro-F1", "std")])
    verdict = "clear difference" if abs(diff) > noise else "within seed noise"
    pairwise.append((exp, a, b, diff, noise, verdict))
    print(f"  {exp}  {SHORT[b]:<9} vs {SHORT[a]:<9}: {diff * 100:+.2f} pp  (std sum {noise * 100:.2f} pp) -> {verdict}")

In [ ]:
# Learning curves per model: mean over seeds (line) with ± 1 std band
epochs_axis = np.arange(1, EPOCHS + 1)
n_cols = 3
n_rows = int(np.ceil(len(MODELS) / n_cols))

for metric, ylabel, scale, fig_name in [("loss", "Cross-entropy loss", 1, "learning_curves_loss"),
                                        ("acc", "Accuracy (%)", 100, "learning_curves_acc")]:
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(17, 3.6 * n_rows), squeeze=False)
    for ax, key in zip(axes.flat, MODELS):
        for split, color, style in [("train", "royalblue", "-"), ("val", "crimson", "--")]:
            curves = np.array([histories[(key, s)][f"{split}_{metric}"] for s in TRAIN_SEEDS]) * scale
            mean, std = curves.mean(0), curves.std(0)
            ax.plot(epochs_axis, mean, color=color, linestyle=style, lw=1.8, label=split)
            ax.fill_between(epochs_axis, mean - std, mean + std, color=color, alpha=0.15)
        ax.axvline(df_runs.loc[df_runs.key == key, "Best epoch"].median(), color="gray", linestyle=":", lw=1.2,
                   label="selected epoch (median)")
        ax.set_title(DISPLAY[key], fontsize=11)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.set_xticks(range(0, EPOCHS + 1, 2))
        ax.grid(True, linestyle=":", alpha=0.6)
        ax.legend(fontsize=8)
    for ax in axes.flat[len(MODELS):]:
        ax.axis("off")
    plt.suptitle(f"{ylabel} per epoch (train metrics in train mode: dropout on, flipped inputs)", fontsize=13, y=1.0)
    plt.tight_layout()
    save_fig(fig_name)
    plt.show()

# Model comparison: validation macro-F1 per epoch and the accuracy / size / speed trade-offs
fig, axes = plt.subplots(1, 3, figsize=(20, 5.2))
for key in MODELS:
    curves = np.array([histories[(key, s)]["val_f1"] for s in TRAIN_SEEDS])
    axes[0].plot(epochs_axis, curves.mean(0), lw=1.8, color=COLORS[key], label=SHORT[key])
    axes[0].fill_between(epochs_axis, curves.mean(0) - curves.std(0), curves.mean(0) + curves.std(0), color=COLORS[key], alpha=0.12)
axes[0].set_title("Validation macro-F1 per epoch (mean ± std)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Macro-F1")
axes[0].legend(fontsize=9, ncol=2)

for ax, xcol, xlabel, scale in [(axes[1], None, "Trainable parameters (log scale)", 1),
                                (axes[2], "Train time (s)", f"Training time for {EPOCHS} epochs (s, log scale)", 1)]:
    for key in MODELS:
        x = param_counts[key] if xcol is None else agg.loc[key, (xcol, "mean")] * scale
        ax.errorbar(x, agg.loc[key, ("Test macro-F1", "mean")], yerr=agg.loc[key, ("Test macro-F1", "std")],
                    fmt="o", capsize=4, markersize=8, color=COLORS[key], label=SHORT[key])
        ax.annotate(SHORT[key], (x, agg.loc[key, ("Test macro-F1", "mean")]), textcoords="offset points", xytext=(6, 4), fontsize=9)
    ax.set_xscale("log")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Test macro-F1")
axes[1].set_title("Test macro-F1 vs. model size")
axes[2].set_title("Test macro-F1 vs. training cost")
for ax in axes:
    ax.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
save_fig("model_comparison")
plt.show()

## Section 9: Error Analysis
- The model for the deep dive is chosen by **mean validation macro-F1**, not by test results.
- Qualitative analysis uses the checkpoint of the first training seed of each model.
- The linear classifier is analysed alongside the best model, so we can see which errors a better representation fixes and which remain.

In [ ]:
best_key = agg[("Val macro-F1", "mean")].idxmax()
ref_seed = TRAIN_SEEDS[0]
print(f"Selected by mean validation macro-F1: {DISPLAY[best_key]} (analysis uses seed {ref_seed})\n")


def load_model(key, seed):
    model = MODEL_SPECS[key][1]().to(device)
    model.load_state_dict(torch.load(checkpoint_path(key, seed), map_location=device, weights_only=True))
    model.eval()
    return model


outputs = {key: evaluate(load_model(key, ref_seed), test_loader, criterion, return_outputs=True) for key in MODELS}
best_out = outputs[best_key]

print(f"Per-class report on the test set ({DISPLAY[best_key]}, seed {ref_seed}):")
print(classification_report(best_out["targets"], best_out["preds"], target_names=CLASS_NAMES, digits=4))

per_class_f1 = pd.DataFrame(
    {SHORT[k]: f1_score(o["targets"], o["preds"], average=None) for k, o in outputs.items()},
    index=CLASS_NAMES,
)
per_class_f1["Gain best vs linear"] = per_class_f1[SHORT[best_key]] - per_class_f1[SHORT["linear"]]
print("Per-class test F1 by model:")
display(per_class_f1.round(4))

fig, ax = plt.subplots(figsize=(14, 5.5))
sns.heatmap(per_class_f1[[SHORT[k] for k in MODELS]].T, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.55, vmax=1.0, ax=ax,
            cbar_kws={"label": "F1"})
ax.set_title(f"Per-class test F1 (seed {ref_seed})", fontsize=13, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
save_fig("per_class_f1")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for ax, key in zip(axes, ["linear", best_key]):
    cm = confusion_matrix(outputs[key]["targets"], outputs[key]["preds"])
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=cm, fmt="d", cmap="Blues", vmin=0, vmax=1, ax=ax, cbar=key == best_key,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(f"{DISPLAY[key]} (counts; color = row-normalized recall)", fontsize=12)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.tick_params(axis="x", rotation=35)
plt.suptitle("Test-set confusion matrices", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig("confusion_matrices")
plt.show()

In [ ]:
# Error groups and most frequent confusion pairs for the best model
y_true, y_pred, probs = best_out["targets"], best_out["preds"], best_out["probs"]
errors = np.where(y_true != y_pred)[0]
print(f"Total test errors: {len(errors)} / {len(y_true)} ({len(errors) / len(y_true) * 100:.2f}%)\n")

UPPER_BODY = {0, 2, 4, 6}   # T-shirt/top, Pullover, Coat, Shirt
FOOTWEAR = {5, 7, 9}        # Sandal, Sneaker, Ankle boot
UPPER_NAME = "Upper-body garment confusion (T-shirt/Pullover/Coat/Shirt)"


def error_group(t, p):
    if t in UPPER_BODY and p in UPPER_BODY:
        return UPPER_NAME
    if t in FOOTWEAR and p in FOOTWEAR:
        return "Footwear confusion (Sandal/Sneaker/Ankle boot)"
    if 3 in (t, p):
        return "Dress vs. other garment"
    return "Other"


def group_counts(key):
    o = outputs[key]
    errs = np.where(o["targets"] != o["preds"])[0]
    return pd.Series([error_group(o["targets"][i], o["preds"][i]) for i in errs]).value_counts(), len(errs)


groups, _ = group_counts(best_key)
display(pd.DataFrame({"Errors": groups, "Share of all errors (%)": (groups / len(errors) * 100).round(1)}))

# Upper-body confusion count for every model: does any representation resolve the dominant error group?
upper_by_model = {}
for key in MODELS:
    g, n_err = group_counts(key)
    upper_by_model[key] = (n_err, int(g.get(UPPER_NAME, 0)))
display(pd.DataFrame({"Model": [SHORT[k] for k in MODELS],
                      "Test errors": [upper_by_model[k][0] for k in MODELS],
                      "Upper-body confusions": [upper_by_model[k][1] for k in MODELS],
                      "Share (%)": [round(upper_by_model[k][1] / upper_by_model[k][0] * 100, 1) for k in MODELS]}))

cm = confusion_matrix(y_true, y_pred)
pairs = [(t, p, cm[t, p]) for t in range(NUM_CLASSES) for p in range(NUM_CLASSES) if t != p and cm[t, p] > 0]
pairs.sort(key=lambda r: -r[2])
top_pairs = pd.DataFrame([{
    "True": CLASS_NAMES[t], "Predicted": CLASS_NAMES[p], "Count": n,
    "% of true class": round(n / cm[t].sum() * 100, 1), "% of all errors": round(n / len(errors) * 100, 1),
} for t, p, n in pairs[:8]])
print("Most frequent confusion pairs:")
display(top_pairs)


def show_grid(rows, title, fig_name):
    """rows: list of (row_label, [test indices]) drawn from the raw test images."""
    n_cols = max(len(idx) for _, idx in rows)
    fig, axes = plt.subplots(len(rows), n_cols, figsize=(1.9 * n_cols, 2.3 * len(rows)), squeeze=False)
    for r, (label, idxs) in enumerate(rows):
        for c in range(n_cols):
            ax = axes[r, c]
            ax.axis("off")
            if c >= len(idxs):
                continue
            i = idxs[c]
            ok = y_true[i] == y_pred[i]
            ax.imshow(test_images[i], cmap="gray")
            ax.set_title(f"T: {CLASS_NAMES[y_true[i]]}\nP: {CLASS_NAMES[y_pred[i]]} ({probs[i, y_pred[i]]:.2f})",
                         fontsize=8, color="darkgreen" if ok else "crimson")
        axes[r, 0].text(-0.15, 0.5, label, transform=axes[r, 0].transAxes, ha="right", va="center", fontsize=9)
    plt.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    save_fig(fig_name)
    plt.show()


rows = []
for t, p, _ in pairs[:3]:
    idx = np.where((y_true == t) & (y_pred == p))[0]
    idx = idx[np.argsort(-probs[idx, p])][:7]
    rows.append((f"{CLASS_NAMES[t]}\n-> {CLASS_NAMES[p]}", idx.tolist()))
show_grid(rows, f"Most confident errors in the top-3 confusion pairs ({DISPLAY[best_key]}; P(prob) = predicted class probability)",
          "errors_top_pairs")

correct = y_true == y_pred
easy, hard = [], []
for c in range(NUM_CLASSES):
    idx = np.where(correct & (y_true == c))[0]
    order = np.argsort(probs[idx, c])
    easy.append(idx[order[-1]])
    hard.append(idx[order[0]])
show_grid([("most confident\ncorrect", easy), ("difficult but\ncorrect", hard)],
          f"Correct predictions per class ({DISPLAY[best_key]})", "correct_examples")

lin_pred = outputs["linear"]["preds"]
fixed = int(np.sum((lin_pred != y_true) & (y_pred == y_true)))
broken = int(np.sum((lin_pred == y_true) & (y_pred != y_true)))
both = int(np.sum((lin_pred != y_true) & (y_pred != y_true)))
print(f"Linear errors fixed by {DISPLAY[best_key]}: {fixed} | new errors introduced: {broken} | errors shared by both: {both}")

## Section 10: Interpretability (Extension)
Each family is inspected through the structure its representation exposes: class templates for the linear model, feature maps for the CNN, the evolution of the hidden state for the LSTM/GRU, and attention weights and positional embeddings for the Transformer.

In [ ]:
# 10.1 Linear classifier weight templates: each row of W reshaped to 28 x 28
linear_weights = load_model("linear", ref_seed).linear.weight.detach().cpu().numpy()

fig, axes = plt.subplots(2, 5, figsize=(15, 6.2))
v_abs = np.abs(linear_weights).max()   # one shared symmetric scale so templates are comparable
for i, ax in enumerate(axes.flat):
    im = ax.imshow(linear_weights[i].reshape(28, 28), cmap="seismic", vmin=-v_abs, vmax=v_abs)
    ax.set_title(f"[{i}] {CLASS_NAMES[i]}", fontsize=11, fontweight="bold")
    ax.axis("off")
fig.subplots_adjust(right=0.9, top=0.86)
cbar = fig.colorbar(im, cax=fig.add_axes([0.92, 0.12, 0.015, 0.72]))
cbar.set_label("Weight (blue < 0 < red)")
fig.suptitle("Linear classifier weight templates $W_k$ (red pixels push toward the class, blue pixels against it)",
             fontsize=13, fontweight="bold")
save_fig("linear_templates")
plt.show()

In [ ]:
# 10.2 CNN: first-layer kernels and feature maps of both stages for one test image
cnn_model = load_model("cnn", ref_seed)
sample_idx = int(np.where(test_labels.numpy() == 6)[0][0])  # first Shirt in the test set
x_sample = test_dataset[sample_idx][0].unsqueeze(0).to(device)
with torch.no_grad():
    after_conv1 = cnn_model.stage1[:3](x_sample)[0].cpu()  # first conv -> BN -> ReLU, 32 x 28 x 28
    after_stage1 = cnn_model.stage1(x_sample)               # 32 x 14 x 14
    after_stage2 = cnn_model.stage2(after_stage1)[0].cpu()  # 64 x 7 x 7
kernels = cnn_model.stage1[0].weight.detach().cpu()[:, 0]   # 32 x 3 x 3

fig = plt.figure(figsize=(16, 9))
grid = fig.add_gridspec(4, 9)
ax = fig.add_subplot(grid[:2, :2])
ax.imshow(test_images[sample_idx], cmap="gray")
ax.set_title(f"Input ({CLASS_NAMES[6]})", fontsize=11)
ax.axis("off")
panels = [("conv1 kernels (3 × 3)", kernels[:7], "coolwarm"),
          ("conv1 feature maps (28 × 28)", after_conv1[:7], "viridis"),
          ("stage-1 output after pooling (14 × 14)", after_stage1[0, :7].cpu(), "viridis"),
          ("stage-2 output after pooling (7 × 7)", after_stage2[:7], "viridis")]
for r, (label, maps, cmap) in enumerate(panels):
    for c in range(7):
        ax = fig.add_subplot(grid[r, 2 + c])
        ax.imshow(maps[c], cmap=cmap)
        ax.axis("off")
        if c == 0:
            ax.set_title(label, fontsize=10, loc="left")
plt.suptitle("CNN: learned kernels and feature maps (first 7 channels of each layer)", fontsize=13, fontweight="bold")
plt.tight_layout()
save_fig("cnn_feature_maps")
plt.show()

In [ ]:
# 10.3 LSTM / GRU: test accuracy when classifying from the hidden state after reading only the first t rows.
# The head was trained on h_28 only, so early-step accuracy is a lower bound on what h_t encodes.
@torch.no_grad()
def prefix_accuracy(model):
    correct_per_step = torch.zeros(28)
    for x, y in test_loader:
        logits = model(x.to(device), all_steps=True)  # [B, 28, 10]
        correct_per_step += (logits.argmax(-1).cpu() == y.unsqueeze(1)).float().sum(0)
    return (correct_per_step / len(test_dataset) * 100).numpy()


prefix_acc = {key: prefix_accuracy(load_model(key, ref_seed)) for key in ["lstm", "gru"]}
rows_axis = np.arange(1, 29)
fig, ax = plt.subplots(figsize=(10, 4.5))
for key, acc in prefix_acc.items():
    ax.plot(rows_axis, acc, marker="o", markersize=3.5, lw=1.8, color=COLORS[key], label=SHORT[key])
ax.axhline(10, color="gray", linestyle=":", lw=1, label="chance (10%)")
ax.set_xlabel("Number of rows read (timestep t)")
ax.set_ylabel("Test accuracy from $h_t$ (%)")
ax.set_title("Recurrent models: accuracy after reading the first t rows of the image")
ax.set_xticks(range(0, 29, 2))
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend()
plt.tight_layout()
save_fig("rnn_prefix_accuracy")
plt.show()

rows_to_95 = {key: int(np.argmax(acc >= 0.95 * acc[-1]) + 1) for key, acc in prefix_acc.items()}
for key, acc in prefix_acc.items():
    print(f"{SHORT[key]}: {acc[13]:.1f}% after 14 rows, {acc[-1]:.1f}% after 28 rows; "
          f"95% of the final accuracy is reached after {rows_to_95[key]} rows")

In [ ]:
# 10.4 Transformer: [CLS] attention of the last block and similarity of the learned positional embeddings
vit_model = load_model("vit", ref_seed)
attn_classes = [0, 2, 4, 6, 3, 9]
attn_idx = [int(np.where((test_labels.numpy() == c) & (outputs["vit"]["preds"] == c))[0][0]) for c in attn_classes]
with torch.no_grad():
    xb_attn = torch.stack([test_dataset[i][0] for i in attn_idx]).to(device)
    vit_model(xb_attn, keep_attn=True)
cls_attn = vit_model.blocks[-1].last_attn[:, 0, 1:].reshape(-1, 7, 7).cpu().numpy()  # [CLS] -> 49 patches

fig, axes = plt.subplots(2, len(attn_idx), figsize=(2.4 * len(attn_idx), 5.2))
for j, i in enumerate(attn_idx):
    axes[0, j].imshow(test_images[i], cmap="gray")
    axes[0, j].set_title(CLASS_NAMES[int(test_labels[i])], fontsize=10)
    axes[1, j].imshow(test_images[i], cmap="gray")
    axes[1, j].imshow(np.kron(cls_attn[j], np.ones((4, 4))), cmap="inferno", alpha=0.6)
for ax in axes.flat:
    ax.axis("off")
axes[1, 0].text(-0.1, 0.5, "[CLS] attention\n(last block)", transform=axes[1, 0].transAxes, ha="right", va="center", fontsize=10)
plt.suptitle("Transformer: where the [CLS] token attends (averaged over heads)", fontsize=13, fontweight="bold")
plt.tight_layout()
save_fig("vit_attention")
plt.show()

pos = F.normalize(vit_model.pos[0, 1:].detach().cpu(), dim=-1)  # 49 patch positions
similarity = (pos @ pos.T).numpy()
ref_positions = [(0, 0), (0, 3), (3, 3), (6, 6)]
fig, axes = plt.subplots(1, len(ref_positions), figsize=(3.2 * len(ref_positions), 3.4))
for ax, (r, c) in zip(axes, ref_positions):
    im = ax.imshow(similarity[r * 7 + c].reshape(7, 7), cmap="viridis", vmin=-1, vmax=1)
    ax.scatter([c], [r], marker="x", color="red", s=60)
    ax.set_title(f"patch at row {r}, col {c}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes, fraction=0.02, label="cosine similarity")
plt.suptitle("Cosine similarity between learned positional embeddings", fontsize=13, fontweight="bold", y=1.04)
save_fig("vit_pos_embedding")
plt.show()

In [ ]:
# Auto-generated key findings, so the discussion is always tied to the numbers of this run
def mean_f1(key):
    return agg.loc[key, ("Test macro-F1", "mean")]


ranking = sorted(MODELS, key=mean_f1, reverse=True)
best_mlp = family_best["MLP"]
worst_cls = per_class_f1[SHORT[best_key]].idxmin()
print("KEY FINDINGS (this run)")
print("- Ranking by mean test macro-F1: " + " > ".join(f"{SHORT[k]} ({mean_f1(k):.4f})" for k in ranking))
print(f"- Best model by validation: {DISPLAY[best_key]}, +{(mean_f1(best_key) - mean_f1('linear')) * 100:.2f} pp over Linear "
      f"and +{(mean_f1(best_key) - mean_f1(best_mlp)) * 100:.2f} pp over the best MLP ({SHORT[best_mlp]}).")
print(f"- CNN vs MLP-C at a similar budget ({param_counts['cnn']:,} vs {param_counts['mlp_c']:,} parameters): "
      f"{(mean_f1('cnn') - mean_f1('mlp_c')) * 100:+.2f} pp.")
print(f"- GRU vs LSTM: {(mean_f1('gru') - mean_f1('lstm')) * 100:+.2f} pp with {param_counts['gru'] / param_counts['lstm'] * 100:.0f}% of the parameters.")
print(f"- Positional encoding (ViT vs ViT-noPE): {(mean_f1('vit') - mean_f1('vit_nopos')) * 100:+.2f} pp.")
print(f"- Hardest class for {SHORT[best_key]}: {worst_cls} (F1 {per_class_f1.loc[worst_cls, SHORT[best_key]]:.4f}); "
      f"upper-body confusions are {groups.get(UPPER_NAME, 0) / len(errors) * 100:.1f}% of its errors.")

### Export for the LaTeX report
Writes every number used by the report to `results/report_data.tex` (LaTeX macros and table rows) and bundles `results/` and `checkpoints/` into `a1_results.zip`. The report never contains hand-typed numbers: copy the extracted `results/` folder into the report directory and recompile.

In [ ]:
import zipfile
from sklearn.metrics import precision_recall_fscore_support


def tex(s):
    s = str(s)
    if "$" in s:  # already LaTeX
        return s
    for a, b in [("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"), ("_", r"\_"), ("#", r"\#"), ("×", r"$\times$")]:
        s = s.replace(a, b)
    return s


def rows(records):
    return "\n".join(" & ".join(tex(v) for v in r) + r" \\" for r in records)


def pm(key, col, digits, scale=1.0):
    mean, std = agg.loc[key, (col, "mean")] * scale, np.nan_to_num(agg.loc[key, (col, "std")]) * scale
    return f"{mean:.{digits}f} $\\pm$ {std:.{digits}f}"


def pp(a, b):
    return f"{(mean_f1(b) - mean_f1(a)) * 100:+.2f}"


CAMEL = {"linear": "Linear", "mlp_a": "MlpA", "mlp_b": "MlpB", "mlp_c": "MlpC", "cnn": "Cnn",
         "lstm": "Lstm", "gru": "Gru", "vit": "Vit", "vit_nopos": "VitNoPos"}
prec, rec, f1s, _ = precision_recall_fscore_support(y_true, y_pred, labels=range(NUM_CLASSES), zero_division=0)

macros = {
    "NormMean": f"{NORM_MEAN:.4f}", "NormStd": f"{NORM_STD:.4f}",
    "TrainN": f"{len(train_idx):,}", "ValN": f"{len(val_idx):,}", "TestN": f"{len(test_idx):,}",
    "BackgroundPct": f"{background_pct:.1f}", "StorageMB": f"{storage_mb:.1f}",
    "DupWithinTrain": dup_stats["within Train"], "DupWithinVal": dup_stats["within Validation"],
    "DupWithinTest": dup_stats["within Test"], "DupTrainVal": dup_stats["Validation in Train"],
    "DupTrainTest": dup_stats["Test in Train"], "DupValTest": dup_stats["Test in Validation"],
    "ImbalanceRatio": f"{imbalance.max():.2f}",
    "NumSeeds": len(TRAIN_SEEDS), "SeedList": ", ".join(map(str, TRAIN_SEEDS)), "Epochs": EPOCHS, "NumModels": len(MODELS),
    "HardwareName": tex(ENV_INFO["gpu"] or f"CPU ({platform.processor() or platform.machine()})"),
    "TotalTrainMinutes": f"{df_runs['Train time (s)'].sum() / 60:.1f}",
    "BestModel": tex(DISPLAY[best_key]), "BestShort": tex(SHORT[best_key]), "RefSeed": ref_seed,
    "BestMlpShort": SHORT[best_mlp],
    "BestTestFOne": pm(best_key, "Test macro-F1", 4), "BestTestAcc": pm(best_key, "Test acc (%)", 2),
    "FOneGainPP": pp("linear", best_key), "GainOverBestMlp": pp(best_mlp, best_key),
    "DeltaCnnMlpC": pp("mlp_c", "cnn"), "DeltaLstmMlpC": pp("mlp_c", "lstm"), "DeltaVitMlpC": pp("mlp_c", "vit"),
    "DeltaGruLstm": pp("lstm", "gru"), "DeltaPosEnc": pp("vit_nopos", "vit"),
    "GruParamPct": f"{param_counts['gru'] / param_counts['lstm'] * 100:.0f}",
    "CnnMlpCParamPct": f"{param_counts['cnn'] / param_counts['mlp_c'] * 100:.0f}",
    "RankingList": ", ".join(f"{SHORT[k]} ({mean_f1(k):.4f})" for k in ranking),
    "WorstClass": tex(worst_cls), "WorstClassFOne": f"{per_class_f1.loc[worst_cls, SHORT[best_key]]:.4f}",
    "TotalErrors": len(errors), "ErrorRate": f"{len(errors) / len(y_true) * 100:.2f}",
    "UpperShare": f"{groups.get(UPPER_NAME, 0) / len(errors) * 100:.1f}",
    "FixedErrors": fixed, "NewErrors": broken, "SharedErrors": both,
    "PrefixHalfLstm": f"{prefix_acc['lstm'][13]:.1f}", "PrefixFullLstm": f"{prefix_acc['lstm'][-1]:.1f}",
    "PrefixHalfGru": f"{prefix_acc['gru'][13]:.1f}", "PrefixFullGru": f"{prefix_acc['gru'][-1]:.1f}",
    "RowsToNinetyFiveLstm": rows_to_95["lstm"], "RowsToNinetyFiveGru": rows_to_95["gru"],
    "ClassDistRows": rows([[c, *dist.loc[c].tolist()] for c in CLASS_NAMES]),
    "EdaStatsRows": rows([[c, f"{eda_class_stats.loc[c, 'mean intensity']:.3f}", f"{eda_class_stats.loc[c, 'foreground ratio']:.3f}"] for c in CLASS_NAMES]),
    "ModelRows": rows([[SHORT[k], FAMILY[k], REPRESENTATION[k], f"{param_counts[k]:,}", ROLE[k]] for k in MODELS]),
    "SummaryRows": rows([[SHORT[k], f"{param_counts[k]:,}", pm(k, "Test acc (%)", 2), pm(k, "Test macro-F1", 4),
                          pm(k, "Val macro-F1", 4), pm(k, "Best epoch", 1), f"{agg.loc[k, ('Gap train-val (pp)', 'mean')]:.2f}"]
                         for k in MODELS]),
    "CostRows": rows([[SHORT[k], f"{param_counts[k]:,}", pm(k, "Train time (s)", 1),
                       pm(k, "Latency bs=1 (ms/img)", 1, scale=1000), pm(k, "Time bs=512 (ms/img)", 2, scale=1000)] for k in MODELS]),
    "FamilyRows": rows([[fam, SHORT[k], pm(k, "Test acc (%)", 2), pm(k, "Test macro-F1", 4), f"{param_counts[k]:,}"]
                        for fam, k in sorted(family_best.items(), key=lambda kv: -mean_f1(kv[1]))]),
    "PairwiseRows": rows([[exp, f"{SHORT[b]} vs. {SHORT[a]}", f"{d * 100:+.2f}", f"{n * 100:.2f}", v] for exp, a, b, d, n, v in pairwise]),
    "PerClassRows": rows([[c, *[f"{per_class_f1.loc[c, SHORT[k]]:.3f}" for k in MODELS], f"{per_class_f1.loc[c, 'Gain best vs linear']:+.3f}"]
                          for c in CLASS_NAMES]),
    "PerClassHeader": " & ".join(rf"\textbf{{{tex(SHORT[k])}}}" for k in MODELS),
    "ClassReportRows": rows([[CLASS_NAMES[c], f"{prec[c]:.4f}", f"{rec[c]:.4f}", f"{f1s[c]:.4f}"] for c in range(NUM_CLASSES)]),
    "ErrorGroupRows": rows([[g, n, f"{n / len(errors) * 100:.1f}"] for g, n in groups.items()]),
    "UpperByModelRows": rows([[SHORT[k], upper_by_model[k][0], upper_by_model[k][1],
                               f"{upper_by_model[k][1] / upper_by_model[k][0] * 100:.1f}"] for k in MODELS]),
    "TopPairRows": rows(top_pairs.values.tolist()),
    "EnvRows": rows([[k, "n/a" if v is None else v] for k, v in ENV_INFO.items() if k not in ("split_seed", "train_seeds")]),
}
for k in MODELS:
    c = CAMEL[k]
    macros[f"TestFOne{c}"] = pm(k, "Test macro-F1", 4)
    macros[f"TestAcc{c}"] = pm(k, "Test acc (%)", 2)
    macros[f"Params{c}"] = f"{param_counts[k]:,}"
    macros[f"TrainTime{c}"] = f"{agg.loc[k, ('Train time (s)', 'mean')]:.1f}"
    macros[f"LatencyUs{c}"] = f"{agg.loc[k, ('Latency bs=1 (ms/img)', 'mean')] * 1000:.1f}"
    macros[f"Gap{c}"] = f"{agg.loc[k, ('Gap train-val (pp)', 'mean')]:.2f}"

with open("./results/report_data.tex", "w", encoding="utf-8") as f:
    f.write("% Auto-generated by 02_architecture_comparison.ipynb. Do not edit by hand.\n")
    for name, value in macros.items():
        f.write(f"\\newcommand{{\\{name}}}{{{value}}}\n")
print(f"Wrote results/report_data.tex ({len(macros)} macros)")

with zipfile.ZipFile("a1_results.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ["results", "checkpoints"]:
        for root, _, files in os.walk(folder):
            for name in files:
                zf.write(os.path.join(root, name))
print(f"Bundled results/ and checkpoints/ into a1_results.zip ({os.path.getsize('a1_results.zip') / 1024**2:.1f} MB)")

try:
    from google.colab import files as colab_files
    colab_files.download("a1_results.zip")
except ImportError:
    print("Not running on Colab: download a1_results.zip manually.")

## Section 11: Discussion, Limitations and Conclusion
*Numbers referenced here come from the tables above and the "KEY FINDINGS" printout. Re-check every sentence after each full run.*

**Overall.** The CNN is the best model by a wide margin. The recurrent models come next and beat every MLP with fewer parameters. The Transformer ends below the larger MLPs under this 20-epoch budget, and the linear classifier is last.

**Representation and inductive bias (E3).**
- *Flattened pixels (Linear, MLP):* no notion of neighbourhood. The linear model underfits with one template per class; the MLP gains a lot from non-linearity (E1), but further width and depth give much smaller gains and a growing train-validation gap (E2).
- *2-D feature maps (CNN):* locality, weight sharing and pooling match the structure of garment images. At a parameter budget similar to MLP-C it is clearly better, and the gain is largest on Pullover, Coat and Shirt, the classes separated by local details.
- *Row sequences (LSTM/GRU):* the vertical order and weights shared across rows are built in, but each row is still a flat vector. This helps moderately: better than any MLP, clearly below the CNN.
- *Patch tokens (Transformer):* global attention from the first layer, but position must be learned. With 50,000 small images and 20 epochs it does not learn locality as efficiently as the CNN has it built in.

**GRU vs. LSTM (E4).** On 28-step sequences the GRU is slightly more accurate than the LSTM with about 75% of the parameters and is faster at inference. The margin passes the decision criterion only narrowly, so the safe conclusion is that the extra cell state of the LSTM brings no benefit here.

**Positional encoding (E5).** Removing it costs several points of macro-F1, most on Dress and Coat: without positions, attention sees a set of patches and cannot tell where the hem or the sleeves are.

**Accuracy vs. cost.** Accuracy does not follow parameter count. Cost is driven by computation per image: convolutions reuse every weight at all positions, recurrent models run 28 sequential steps, and attention mixes 50 tokens in every block. At batch size 512 the CNN is the most expensive model per image. At batch size 1 latency is dominated by the number of sequential kernel launches, so the Transformer is the slowest, while the LSTM/GRU are faster than the CNN because cuDNN runs the whole recurrence as one fused kernel. All models stay below 2 ms per image, so the choice is driven by accuracy.

**Error analysis.** The confusion among T-shirt/top, Pullover, Coat and Shirt is the largest error group for every model. Better representations reduce the number of these errors, but their share stays high, which points to the resolution limit and label ambiguity of the dataset. Shirt is the hardest class for every model.

**Limitations.**
- Only Fashion-MNIST, one or two small architectures per family, three seeds, 20 epochs and shared untuned hyperparameters; the Transformer in particular is known to benefit from longer training and stronger augmentation.
- Only horizontal flips are used for augmentation.
- Some GPU kernels are not bit-exact, so repeated GPU runs can differ slightly.
- Duplicates are detected by exact hashing only.

**Conclusion.** For small grayscale images and a small training budget, the amount of spatial structure an architecture builds in matters more than its number of parameters, and a compact CNN gives the best balance of accuracy, size and cost.

## AI Usage
The use of AI tools in this project is documented in `AI_USAGE.md` in the repository root.